# Ocean Climate AI — NOAA Next-Day Prediction
### SI Lab practicals + trained models + Streamlit interface

This notebook collects NOAA historical data, performs statistical practicals, trains and evaluates five classification and five regression models, and saves artifacts for a **separate Streamlit application**.

**Run all cells**, then download the model and three CSV files. Put them beside `app.py` from the accompanying Streamlit project ZIP. The Streamlit app calls the trained models to predict tomorrow's NOAA-style alert and DHW.

**Scope:** Educational next-day heat-stress forecasting for historical Hawaiian regional data. It does not directly predict observed coral bleaching or mortality.


In [ ]:
!pip -q install joblib scikit-learn seaborn scipy


## 1. Download official NOAA data

In [ ]:
import requests, re, io, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

SOURCE = "https://www.coralreefwatch.noaa.gov/product/vs/data/hawaii.txt"
response = requests.get(SOURCE, timeout=45, headers={"User-Agent": "Mozilla/5.0"})
response.raise_for_status()
raw = response.text
print("NOAA file downloaded:", len(raw), "characters")
print("Preview of data lines:")
for line in raw.splitlines():
    if re.match(r"^\s*(?:19|20)\d{2}\s+\d{1,2}\s+\d{1,2}\s+", line):
        print(line[:170]); break

NOAA's regional station file contains 10 fields per daily observation: year, month, day, SST minimum, SST maximum, representative SST, SST anomaly, HotSpot, DHW, and seven-day maximum alert code. The parsing cell checks this layout instead of silently making up missing data.

In [ ]:
rows = []
for line in raw.splitlines():
    parts = re.split(r"\s+", line.strip())
    if len(parts) != 10 or not re.fullmatch(r"(?:19|20)\d{2}", parts[0]):
        continue
    try:
        vals = [float(x) for x in parts]
        if not (1 <= vals[1] <= 12 and 1 <= vals[2] <= 31):
            continue
        rows.append(vals)
    except ValueError:
        pass

columns = ["Year", "Month", "Day", "SST_Min", "SST_Max", "SST", "SST_Anomaly", "HotSpot", "DHW", "Alert"]
if len(rows) < 500:
    raise ValueError("NOAA data format may have changed. Please inspect the file preview before proceeding. No synthetic data will be used.")
df = pd.DataFrame(rows, columns=columns)
df["Date"] = pd.to_datetime(df[["Year", "Month", "Day"]].astype(int), errors="coerce")
for c in ["SST_Min", "SST_Max", "SST", "SST_Anomaly", "HotSpot", "DHW", "Alert"]:
    df.loc[df[c] <= -900, c] = np.nan
# Alert code is defined by NOAA as 0 through 4 for this regional product.
df.loc[~df["Alert"].isin([0, 1, 2, 3, 4]), "Alert"] = np.nan
df = df.dropna(subset=["Date"]).sort_values("Date").drop_duplicates("Date").reset_index(drop=True)
print("Downloaded rows:", len(df), "| Range:", df.Date.min().date(), "to", df.Date.max().date())
display(df.tail())

## 2. Preprocess and define the prediction target

In [ ]:
# Use today's satellite measurements to predict TOMORROW's alert and DHW.
FEATURES = ["SST", "SST_Anomaly", "HotSpot", "DHW"]
df["Next_Date"] = df["Date"].shift(-1)
df["Next_Alert"] = df["Alert"].shift(-1)
df["Next_DHW"] = df["DHW"].shift(-1)
# Only true consecutive-day pairs are valid forecasting examples.
data = df.loc[(df["Next_Date"] - df["Date"]).dt.days.eq(1)].copy()
data = data.dropna(subset=FEATURES + ["Next_Alert", "Next_DHW"])
data["Next_Alert"] = data["Next_Alert"].astype(int)
if len(data) < 300 or data["Next_Alert"].nunique() < 2:
    raise ValueError("Not enough valid labeled daily observations for model training.")
print("Valid next-day examples:", len(data))
print("Next-day class distribution:")
display(data["Next_Alert"].value_counts().sort_index().rename("Count"))
print("Missing values:", data[FEATURES + ["Next_Alert", "Next_DHW"]].isna().sum().sum())
data.to_csv("noaa_hawaii_training_data.csv", index=False)

## 3. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
data["Next_Alert"].value_counts().sort_index().plot.bar(ax=ax[0], color="#ef8e38")
ax[0].set(title="Next-day NOAA alert distribution", xlabel="Alert code (0–4)", ylabel="Daily observations")
ax[1].plot(data["Date"], data["DHW"], color="#087f8c", lw=.6)
ax[1].set(title="Historical Degree Heating Weeks", xlabel="Date", ylabel="DHW (°C-weeks)")
plt.tight_layout(); plt.show()
print("DHW mean:", round(data.DHW.mean(), 3), "| Median:", round(data.DHW.median(), 3))
print("DHW standard deviation:", round(data.DHW.std(), 3))

## Practical 2 — Data exploration and descriptive statistics
We use **actual NOAA daily measurements** (DHW, SST, anomaly), not invented stress scores. These statistics describe the dataset and do not constitute future predictions.

In [ ]:
print("Rows and columns:", data.shape)
print("Missing values:\n", data[FEATURES].isna().sum())
print("Duplicate dates:", data["Date"].duplicated().sum())
print("\nDescriptive statistics:")
display(data[FEATURES].describe().round(3))
x = data["DHW"].dropna()
print("Mean:", round(x.mean(), 3), "Median:", round(x.median(), 3))
print("Mode:", x.mode().iloc[0], "Variance:", round(x.var(), 3))
print("Standard deviation:", round(x.std(), 3))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(x, bins=30, color="#0c8b91", edgecolor="white")
axes[0].set(title="DHW distribution", xlabel="DHW (°C-weeks)", ylabel="Frequency")
axes[1].boxplot(x)
axes[1].set(title="DHW boxplot", ylabel="DHW (°C-weeks)")
plt.tight_layout(); plt.show()

## Practical 3 — Probability and sampling
Estimate empirical probabilities from observed days, then draw a reproducible random sample. These are historical proportions, not forecast probabilities.

In [ ]:
print("P(next-day alert >= 3):", round((data["Next_Alert"] >= 3).mean(), 4))
print("P(next-day no stress):", round((data["Next_Alert"] == 0).mean(), 4))
sample = data["DHW"].sample(n=min(100, len(data)), random_state=42)
print("Sample size:", len(sample), "Sample mean DHW:", round(sample.mean(), 3))
print("Population (dataset) mean DHW:", round(data["DHW"].mean(), 3))

## Practical 4 — Central Limit Theorem (CLT)
Repeatedly sample daily DHW measurements to illustrate the distribution of sample means. **Caution:** daily observations are temporally correlated; this simple demonstration treats sampled days as independent for teaching purposes.

In [ ]:
rng = np.random.default_rng(42)
values = data["DHW"].to_numpy()
means = [rng.choice(values, size=30, replace=True).mean() for _ in range(500)]
plt.figure(figsize=(8, 4))
plt.hist(means, bins=25, color="#0b8493", edgecolor="white")
plt.axvline(np.mean(values), color="#e77d32", linestyle="--", label="Full-data mean")
plt.title("CLT demonstration — 500 sample means (n=30)")
plt.xlabel("Sample mean DHW"); plt.ylabel("Frequency"); plt.legend(); plt.show()
print("Mean of sample means:", round(np.mean(means), 3))

## Practical 5 — Confidence interval, T-test and illustrative Z-test
For a classroom example, test the observed mean DHW against a **stated reference value of 4 °C-weeks**. This benchmark is chosen for illustration, not a scientifically established population mean. Ordinary tests assume independent observations; daily time-series dependence makes these p-values illustrative rather than rigorous environmental inference.

In [ ]:
from scipy import stats
from scipy.stats import norm
x = data["DHW"].dropna().to_numpy()
n_obs = len(x)
mean = np.mean(x)
se = stats.sem(x)
ci = stats.t.interval(0.95, df=n_obs - 1, loc=mean, scale=se)
t_stat, t_p = stats.ttest_1samp(x, popmean=4.0)
# Z-test example with ASSUMED known population standard deviation = 2
assumed_sigma = 2.0
z_stat = (mean - 4.0) / (assumed_sigma / np.sqrt(n_obs))
z_p = 2 * norm.sf(abs(z_stat))
print("95% confidence interval for mean DHW:", tuple(round(v, 4) for v in ci))
print("One-sample T-test vs mean 4: t=%.4f, p=%.6g" % (t_stat, t_p))
print("Illustrative Z-test (assumed sigma=2): z=%.4f, p=%.6g" % (z_stat, z_p))
print("Decision (T-test, alpha=0.05):", "Reject H0" if t_p < .05 else "Fail to reject H0")

## Practical 6 — Chi-square and one-way ANOVA
Chi-square compares **same-day temperature-anomaly sign** with **same-day NOAA alert status**. ANOVA compares DHW means across calendar months. Because NOAA alerts derive from heat-stress measurements and observations are time-correlated, these are demonstrations of statistical methods, not independent causal evidence.

In [ ]:
from scipy.stats import chi2_contingency, f_oneway
stat_df = df.dropna(subset=["Date", "SST_Anomaly", "DHW", "Alert"]).copy()
stat_df["Positive_Anomaly"] = np.where(stat_df["SST_Anomaly"] > 0, "Positive", "Non-positive")
stat_df["High_Alert"] = np.where(stat_df["Alert"] >= 3, "High", "Lower")
contingency = pd.crosstab(stat_df["Positive_Anomaly"], stat_df["High_Alert"])
display(contingency)
if contingency.shape == (2, 2):
    chi2, p, dof, expected = chi2_contingency(contingency)
    print("Chi-square = %.4f, p = %.6g" % (chi2, p))
else:
    print("Chi-square skipped: both categories must be present.")
month_groups = [g["DHW"].to_numpy() for _, g in stat_df.groupby(stat_df["Date"].dt.month) if len(g) >= 2]
if len(month_groups) >= 2:
    f, p_anova = f_oneway(*month_groups)
    print("One-way ANOVA (monthly DHW): F = %.4f, p = %.6g" % (f, p_anova))
else:
    print("ANOVA skipped: insufficient monthly groups.")

## Additional practical — Correlation and covariance
Examine relationships between numerical environmental measurements. Correlation does not establish causation.

In [ ]:
print("Pearson correlation:")
display(data[FEATURES].corr(method="pearson").round(3))
print("Spearman correlation:")
display(data[FEATURES].corr(method="spearman").round(3))
print("Covariance:")
display(data[FEATURES].cov().round(3))
plt.figure(figsize=(7, 5))
sns.heatmap(data[FEATURES].corr(), annot=True, cmap="BrBG", center=0)
plt.title("Environmental feature correlations"); plt.tight_layout(); plt.show()

## 4. Chronological training and test split

In [ ]:
# Do not shuffle time-series data: future records must never train the past.
split = int(len(data) * 0.80)
train, test = data.iloc[:split].copy(), data.iloc[split:].copy()
X_train, X_test = train[FEATURES], test[FEATURES]
y_train_cls, y_test_cls = train["Next_Alert"], test["Next_Alert"]
y_train_reg, y_test_reg = train["Next_DHW"], test["Next_DHW"]
print("Training:", len(train), "rows through", train.Date.max().date())
print("Testing:", len(test), "rows from", test.Date.min().date())
print("Test classes:", sorted(y_test_cls.unique()))

## 5. Train and compare classification models

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone
import joblib

classifiers = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1200, class_weight="balanced")),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=5, random_state=42, n_jobs=-1),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9)),
    "SVM": make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
}
classification_rows = []
fitted_classifiers = {}
for name, model in classifiers.items():
    model.fit(X_train, y_train_cls)
    pred = model.predict(X_test)
    p, r, f1, _ = precision_recall_fscore_support(y_test_cls, pred, average="macro", zero_division=0)
    classification_rows.append({"Model": name, "Accuracy": accuracy_score(y_test_cls, pred),
                                "Macro Precision": p, "Macro Recall": r, "Macro F1": f1})
    fitted_classifiers[name] = model
classification_results = pd.DataFrame(classification_rows).sort_values("Macro F1", ascending=False)
display(classification_results.round(3))
classification_results.to_csv("classification_results.csv", index=False)
BEST_CLASS_NAME = classification_results.iloc[0]["Model"]
best_classifier = fitted_classifiers[BEST_CLASS_NAME]
print("Selected classifier (highest test macro F1):", BEST_CLASS_NAME)
print(classification_report(y_test_cls, best_classifier.predict(X_test), zero_division=0))

## 6. Train and compare regression models

In [ ]:
regressors = {
    "Linear Regression": LinearRegression(),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso": make_pipeline(StandardScaler(), Lasso(alpha=0.01, max_iter=10000)),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=8, min_samples_leaf=10, random_state=42),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_leaf=5, random_state=42, n_jobs=-1),
}
regression_rows, fitted_regressors = [], {}
for name, model in regressors.items():
    model.fit(X_train, y_train_reg)
    pred = model.predict(X_test)
    regression_rows.append({"Model": name, "MAE": mean_absolute_error(y_test_reg, pred),
                            "RMSE": np.sqrt(mean_squared_error(y_test_reg, pred)),
                            "R2": r2_score(y_test_reg, pred)})
    fitted_regressors[name] = model
regression_results = pd.DataFrame(regression_rows).sort_values("RMSE")
display(regression_results.round(3))
regression_results.to_csv("regression_results.csv", index=False)
BEST_REG_NAME = regression_results.iloc[0]["Model"]
best_regressor = fitted_regressors[BEST_REG_NAME]
print("Selected regressor (lowest test RMSE):", BEST_REG_NAME)

## 7. Evaluation charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
labels = [0, 1, 2, 3, 4]
cm = confusion_matrix(y_test_cls, best_classifier.predict(X_test), labels=labels)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0], xticklabels=labels, yticklabels=labels)
axes[0].set(title=f"Confusion matrix — {BEST_CLASS_NAME}", xlabel="Predicted next-day alert", ylabel="Actual next-day alert")
reg_pred = best_regressor.predict(X_test)
axes[1].scatter(y_test_reg, reg_pred, s=10, alpha=.3, color="#ef8e38")
axes[1].set(title=f"Next-day DHW — {BEST_REG_NAME}", xlabel="Actual DHW", ylabel="Predicted DHW")
plt.tight_layout(); plt.show()

## 8. Save the trained models

In [ ]:
artifact = {"classifier": best_classifier, "regressor": best_regressor,
            "features": FEATURES, "classification_model": BEST_CLASS_NAME,
            "regression_model": BEST_REG_NAME,
            "training_region": "Main Hawaiian Islands", "target_horizon": "next day"}
joblib.dump(artifact, "ocean_climate_models.joblib")
print("Saved ocean_climate_models.joblib")

## 9. Export artifacts for the Streamlit application


In [ ]:
# Download these files and place them next to app.py in the Streamlit project.
from google.colab import files

export_files = [
    "ocean_climate_models.joblib",
    "classification_results.csv",
    "regression_results.csv",
    "noaa_hawaii_training_data.csv",
]
for filename in export_files:
    if os.path.exists(filename):
        print(f"Ready: {filename} ({os.path.getsize(filename):,} bytes)")
    else:
        print(f"MISSING: {filename}")

# Uncomment these lines to download each artifact:
# for filename in export_files:
#     files.download(filename)


## 10. Streamlit demo and submission

1. Run all notebook cells to generate the trained model and evaluation CSVs.
2. Download the four artifacts listed above (uncomment `files.download` if needed).
3. Unzip the accompanying **Ocean_Climate_Streamlit_Project.zip**.
4. Copy the four artifacts into the same folder as `app.py`.
5. Run locally: `pip install -r requirements.txt` then `python -m streamlit run app.py`.
6. For an online demonstration, upload the app, dependencies and generated files to GitHub and deploy with **Streamlit Community Cloud**.
7. Test an actual prediction, take screenshots of the results and update your SI report with **new** model metrics.

**Important:** This notebook compares models on the holdout period and chooses the best based on those results. For a stronger scientific evaluation, select models using a separate validation period and reserve the test period for final reporting.

**Do not claim:** This system predicts observed coral death or achieves 100% future forecasting accuracy.
